# P22: Down syndrome fetal cortex Multiome, one-notebook workflow

Dataset: Single-Cell Multiomic Atlas of Human Cortical Development in Down Syndrome
CELLxGENE collection: 0e9fd1d3-ef4c-47c6-a2e4-ef4bfadf7c79
CELLxGENE dataset: f16c25da-15bd-46a4-9a3f-17093f27a2f1
GEO: GSE305153

Safe default run fetches live public catalog metadata, validates dataset identity and expected scale, and runs a small synthetic five-seed model-wiring test. It does not load disease cells.

Post-approval run downloads the processed H5AD into local cache, validates file-level metadata and RNA/ATAC evidence, then runs donor-held-out baselines, gated fusion, and routing interventions.

Set P22_PROFESSOR_APPROVED=1, P22_RUN_REAL_DATA=1, and P22_RUN_MODEL=1 only after Professor Fang approves this direction and file-level preflight passes. H5AD and fragment files are not committed because they are large.

Sources: Nature Medicine study: https://www.nature.com/articles/s41591-026-04211-1
PubMed: https://pubmed.ncbi.nlm.nih.gov/41545595/
CELLxGENE: https://cellxgene.cziscience.com/collections/0e9fd1d3-ef4c-47c6-a2e4-ef4bfadf7c79


In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
from pathlib import Path

os.environ.setdefault('PYTHONDONTWRITEBYTECODE', '1')

import numpy as np
import pandas as pd
import requests

SEED = 20260728
COLLECTION_ID = '0e9fd1d3-ef4c-47c6-a2e4-ef4bfadf7c79'
DATASET_ID = 'f16c25da-15bd-46a4-9a3f-17093f27a2f1'
CELLXGENE_API = 'https://api.cellxgene.cziscience.com/dp/v1'
EXPECTED_CELLS = 248_998
EXPECTED_DONORS = 30
EXPECTED_H5AD_BYTES = 1_569_658_860

PROFESSOR_APPROVED = os.getenv('P22_PROFESSOR_APPROVED', '0') == '1'
RUN_REAL_DATA = os.getenv('P22_RUN_REAL_DATA', '0') == '1'
RUN_MODEL = os.getenv('P22_RUN_MODEL', '0') == '1'
if (RUN_REAL_DATA or RUN_MODEL) and not PROFESSOR_APPROVED:
    raise RuntimeError(
        'Approval gate: set P22_PROFESSOR_APPROVED=1 only after Professor Fang approval.'
    )

PROJECT_ROOT = Path.cwd()
DATA_ROOT = Path(os.getenv('P22_DATA_ROOT', PROJECT_ROOT / 'data' / 'real'))
OUTPUT_ROOT = Path(os.getenv('P22_OUTPUT_ROOT', PROJECT_ROOT / 'reports' / 'generated' / 'down_syndrome'))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print({
    'python': sys.version.split()[0],
    'project_root': str(PROJECT_ROOT),
    'data_root': str(DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'professor_approved': PROFESSOR_APPROVED,
    'run_real_data': RUN_REAL_DATA,
    'run_model': RUN_MODEL,
})


In [ ]:
def get_json(url):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return response.json()

collection = get_json(f'{CELLXGENE_API}/collections/{COLLECTION_ID}')
dataset = next(item for item in collection['datasets'] if item['id'] == DATASET_ID)
h5ad_asset = next(item for item in dataset['dataset_assets'] if item['filetype'] == 'H5AD')
fragment_asset = next(
    (item for item in dataset['dataset_assets'] if item['filetype'] == 'ATAC_FRAGMENT'),
    None,
)

catalog_summary = {
    'collection_name': collection['name'],
    'dataset_name': dataset['name'],
    'dataset_id': dataset['id'],
    'assay': [item['label'] for item in dataset['assay']],
    'cells': dataset['cell_count'],
    'donors': len(dataset['donor_id']),
    'conditions': [item['label'] for item in dataset['disease']],
    'stages': len(dataset['development_stage']),
    'h5ad_url': f'https://datasets.cellxgene.cziscience.com/{DATASET_ID}.h5ad',
    'fragment_asset_present': fragment_asset is not None,
}
print(json.dumps(catalog_summary, indent=2))


In [ ]:
# Metadata-only gate. No cell-level data loaded.
assert dataset['id'] == DATASET_ID
assert dataset['name'].startswith('Human fetal cortex in Down syndrome')
assert dataset['cell_count'] == EXPECTED_CELLS
assert len(dataset['donor_id']) == EXPECTED_DONORS
assert [item['label'] for item in dataset['assay']] == ['10x multiome']
assert {item['label'] for item in dataset['disease']} == {'complete trisomy 21', 'normal'}
assert h5ad_asset['filetype'] == 'H5AD'
assert h5ad_asset['s3_uri'].endswith('/local.h5ad')

condition_counts = {
    'normal': sum('_CON_' in donor for donor in dataset['donor_id']),
    'complete trisomy 21': sum('_DS_' in donor for donor in dataset['donor_id']),
}
assert condition_counts == {'normal': 15, 'complete trisomy 21': 15}

metadata_gate = {
    'status': 'PASS',
    'dataset_identity': 'PASS',
    'assay': '10x multiome',
    'cells': EXPECTED_CELLS,
    'donors': EXPECTED_DONORS,
    'condition_counts': condition_counts,
    'developmental_stages': len(dataset['development_stage']),
    'file_level_pairing': 'PENDING: requires H5AD inspection',
}
print(json.dumps(metadata_gate, indent=2))


## Local resource plan

Processed H5AD is about 1.57 GB. ATAC fragment asset is separate and much larger. First local run uses backed H5AD reads and a fixed per-donor cell cap. It does not load fragment files.

Real-data output is not evidence until donor IDs, condition, stage, sex, batch, RNA/ATAC blocks, and same-nucleus linkage pass file-level checks.


In [ ]:
resource_plan = {
    'processed_h5ad_gb': round((h5ad_asset.get('file_size') or EXPECTED_H5AD_BYTES) / 1e9, 3),
    'fragment_asset_present': fragment_asset is not None,
    'first_real_run': 'backed H5AD plus stratified per-donor subset',
    'raw_fragments_loaded': False,
    'real_model_default': 'OFF',
    'approval_required_for_real_model': True,
}
print(json.dumps(resource_plan, indent=2))


## Safe model-wiring test

Synthetic only. Verifies donor-held-out splitting, train-only feature selection/scaling, five seeds, gated fusion, and intervention plumbing. This is not a Down syndrome result.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn

def make_demo_data(seed=SEED, donors_per_class=6, cells_per_donor=24):
    rng = np.random.default_rng(seed)
    donors, labels = [], []
    for label, prefix in [(0, 'CON'), (1, 'DS')]:
        for donor_no in range(donors_per_class):
            donor = f'DEMO_{prefix}_{donor_no:02d}'
            for _ in range(cells_per_donor):
                donors.append(donor)
                labels.append(label)
    n = len(donors)
    x_rna = rng.normal(size=(n, 32)).astype('float32')
    x_atac = rng.normal(size=(n, 32)).astype('float32')
    x_rna[:, :4] += np.asarray(labels)[:, None] * 0.8
    x_atac[:, 4:8] += np.asarray(labels)[:, None] * 0.7
    return x_rna, x_atac, np.asarray(labels), np.asarray(donors)

def donor_split(donors, labels, seed=SEED):
    rng = np.random.default_rng(seed)
    donor_frame = pd.DataFrame({'donor': donors, 'label': labels}).drop_duplicates()
    parts = {'train': [], 'val': [], 'test': []}
    for _, group in donor_frame.groupby('label', sort=True):
        values = group.donor.to_numpy().copy()
        rng.shuffle(values)
        n_test = max(1, round(len(values) * 0.2))
        n_val = max(1, round(len(values) * 0.2))
        parts['test'].extend(values[:n_test])
        parts['val'].extend(values[n_test:n_test + n_val])
        parts['train'].extend(values[n_test + n_val:])
    masks = {name: np.isin(donors, values) for name, values in parts.items()}
    assert not (set(parts['train']) & set(parts['val']))
    assert not (set(parts['train']) & set(parts['test']))
    assert not (set(parts['val']) & set(parts['test']))
    return masks, parts

def fit_train_only_view(x, train_mask, n_features=16):
    x = np.asarray(x, dtype='float32')
    variance = np.var(x[train_mask], axis=0)
    selected = np.argsort(variance)[::-1][:min(n_features, x.shape[1])]
    scaler = StandardScaler().fit(x[train_mask][:, selected])
    return selected, scaler

def transform_view(x, selected, scaler):
    return scaler.transform(np.asarray(x)[:, selected]).astype('float32')

class GatedFusion(nn.Module):
    # Routing MLP plus weighted branch sum; not QKV cross-attention.
    def __init__(self, dim_a, dim_b, n_classes=2, embed_dim=24):
        super().__init__()
        self.encoder_a = nn.Sequential(nn.Linear(dim_a, embed_dim), nn.ReLU())
        self.encoder_b = nn.Sequential(nn.Linear(dim_b, embed_dim), nn.ReLU())
        self.gate = nn.Sequential(nn.Linear(embed_dim * 2, 16), nn.ReLU(), nn.Linear(16, 2))
        self.head = nn.Linear(embed_dim, n_classes)

    def forward(self, x_a, x_b, route_override=None, ablate=None):
        z_a = self.encoder_a(x_a)
        z_b = self.encoder_b(x_b)
        if ablate == 'rna':
            z_a = torch.zeros_like(z_a)
        if ablate == 'atac':
            z_b = torch.zeros_like(z_b)
        if route_override is None:
            weights = torch.softmax(self.gate(torch.cat([z_a, z_b], dim=1)), dim=1)
        else:
            fixed = torch.tensor(route_override, dtype=z_a.dtype, device=z_a.device)
            weights = fixed.unsqueeze(0).expand(z_a.shape[0], 2)
        fused = weights[:, :1] * z_a + weights[:, 1:] * z_b
        return self.head(fused), weights

def accuracy_report(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }

def fit_gated(x_a, x_b, y, train_mask, val_mask, seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    model = GatedFusion(x_a.shape[1], x_b.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
    xa, xb = torch.tensor(x_a), torch.tensor(x_b)
    yt = torch.tensor(y, dtype=torch.long)
    best_state, best_val = None, -np.inf
    for _ in range(40):
        model.train()
        optimizer.zero_grad()
        logits, _ = model(xa[train_mask], xb[train_mask])
        nn.functional.cross_entropy(logits, yt[train_mask]).backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_logits, _ = model(xa[val_mask], xb[val_mask])
        score = float((val_logits.argmax(1) == yt[val_mask]).float().mean())
        if score > best_val:
            best_val = score
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model


In [ ]:
def run_demo(seed):
    x_rna, x_atac, y, donors = make_demo_data(seed=seed)
    masks, donor_parts = donor_split(donors, y, seed=seed)
    rna_selected, rna_scaler = fit_train_only_view(x_rna, masks['train'])
    atac_selected, atac_scaler = fit_train_only_view(x_atac, masks['train'])
    rna = transform_view(x_rna, rna_selected, rna_scaler)
    atac = transform_view(x_atac, atac_selected, atac_scaler)
    results = []
    for name, matrix in [('rna_only', rna), ('atac_only', atac), ('concat', np.hstack([rna, atac]))]:
        model = LogisticRegression(max_iter=250, random_state=seed).fit(matrix[masks['train']], y[masks['train']])
        pred = model.predict(matrix[masks['test']])
        results.append({'seed': seed, 'model': name, 'intervention': 'normal', **accuracy_report(y[masks['test']], pred)})
    gated = fit_gated(rna, atac, y, masks['train'], masks['val'], seed)
    gated.eval()
    xa, xb = torch.tensor(rna[masks['test']]), torch.tensor(atac[masks['test']])
    yt = y[masks['test']]
    with torch.no_grad():
        interventions = [
            ('normal', {}),
            ('route_rna_only', {'route_override': [1.0, 0.0]}),
            ('route_atac_only', {'route_override': [0.0, 1.0]}),
            ('ablate_rna', {'ablate': 'rna'}),
            ('ablate_atac', {'ablate': 'atac'}),
        ]
        for intervention, kwargs in interventions:
            logits, weights = gated(xa, xb, **kwargs)
            pred = logits.argmax(1).numpy()
            row = {'seed': seed, 'model': 'gated', 'intervention': intervention, **accuracy_report(yt, pred)}
            row['mean_rna_route'] = float(weights[:, 0].mean())
            row['mean_atac_route'] = float(weights[:, 1].mean())
            results.append(row)
    return results, donor_parts

demo_rows = []
for demo_seed in range(5):
    rows, donor_parts = run_demo(SEED + demo_seed)
    demo_rows.extend(rows)

demo_results = pd.DataFrame(demo_rows)
assert len(demo_results) == 5 * 8
gated_rows = demo_results[demo_results.model == 'gated']
assert np.allclose(gated_rows.mean_rna_route + gated_rows.mean_atac_route, 1.0, atol=1e-5)
assert not (set(donor_parts['train']) & set(donor_parts['test']))
demo_results.to_csv(OUTPUT_ROOT / 'demo_model_results.csv', index=False)
print('DEMO PASS', demo_results.shape)
print(demo_results.groupby('model')[['accuracy', 'macro_f1']].mean())


## Post-approval real-data path

After approval, run:

P22_PROFESSOR_APPROVED=1 P22_RUN_REAL_DATA=1 P22_RUN_MODEL=1 jupyter lab P22_down_syndrome_all_in_one.ipynb

The real path downloads processed H5AD, opens it backed, checks donor/condition/stage/sex/batch fields, tests RNA and ATAC feature evidence, samples a fixed number of cells per donor, and only then permits model fitting. Missing same-nucleus ATAC evidence stops the run.


In [ ]:
def download_h5ad(destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size == EXPECTED_H5AD_BYTES:
        return destination
    url = f'https://datasets.cellxgene.cziscience.com/{DATASET_ID}.h5ad'
    temporary = destination.with_suffix(destination.suffix + '.partial')
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with temporary.open('wb') as handle:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    handle.write(chunk)
    if temporary.stat().st_size != EXPECTED_H5AD_BYTES:
        temporary.unlink(missing_ok=True)
        raise RuntimeError('Downloaded H5AD size does not match catalog size')
    temporary.replace(destination)
    return destination

def resolve_obs_column(obs, candidates):
    for candidate in candidates:
        if candidate in obs.columns:
            return candidate
    return None

def matrix_to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, 'toarray') else np.asarray(matrix)

def inspect_h5ad(path):
    import anndata as ad
    backed = ad.read_h5ad(path, backed='r')
    obs = backed.obs
    fields = {
        'donor': resolve_obs_column(obs, ['donor_id', 'donor', 'sample_id', 'sample']),
        'condition': resolve_obs_column(obs, ['disease', 'condition', 'group']),
        'stage': resolve_obs_column(obs, ['development_stage', 'stage', 'gestational_week']),
        'sex': resolve_obs_column(obs, ['sex']),
        'batch': resolve_obs_column(obs, ['batch', 'library_id', 'sample_id']),
        'cell_type': resolve_obs_column(obs, ['cell_type', 'celltype', 'cluster_name']),
    }
    missing = [name for name in ('donor', 'condition', 'stage') if fields[name] is None]
    feature_types = backed.var['feature_types'].astype(str).str.lower() if 'feature_types' in backed.var else None
    atac_obsm = [key for key in backed.obsm_keys() if any(token in key.lower() for token in ('atac', 'peak'))]
    has_rna = bool(feature_types is not None and feature_types.str.contains('gene|rna|expression', regex=True).any())
    has_atac = bool(feature_types is not None and feature_types.str.contains('peak|atac', regex=True).any()) or bool(atac_obsm)
    summary = {
        'shape': tuple(map(int, backed.shape)),
        'obs_columns': list(obs.columns),
        'fields': fields,
        'missing_required': missing,
        'feature_type_values': sorted(feature_types.unique().tolist()) if feature_types is not None else [],
        'atac_obsm_keys': atac_obsm,
        'rna_evidence': has_rna,
        'atac_evidence': has_atac,
        'pairing_status': 'PASS' if not missing and has_rna and has_atac else 'STOP',
    }
    backed.file.close()
    return summary

def sample_cells_by_donor(path, max_cells_per_donor=256, seed=SEED):
    import anndata as ad
    backed = ad.read_h5ad(path, backed='r')
    obs = backed.obs.copy()
    donor_col = resolve_obs_column(obs, ['donor_id', 'donor', 'sample_id', 'sample'])
    condition_col = resolve_obs_column(obs, ['disease', 'condition', 'group'])
    if donor_col is None or condition_col is None:
        backed.file.close()
        raise RuntimeError('Required donor/condition fields missing')
    selected = []
    rng = np.random.default_rng(seed)
    for _, rows in obs.groupby(donor_col, observed=True):
        values = rows.index.to_numpy()
        if len(values) > max_cells_per_donor:
            values = rng.choice(values, size=max_cells_per_donor, replace=False)
        selected.extend(values.tolist())
    sampled = backed[selected].to_memory()
    backed.file.close()
    return sampled, donor_col, condition_col

def extract_rna_atac(adata):
    if 'feature_types' in adata.var:
        feature_types = adata.var['feature_types'].astype(str).str.lower()
        rna_mask = feature_types.str.contains('gene|rna|expression', regex=True).to_numpy()
        atac_mask = feature_types.str.contains('peak|atac', regex=True).to_numpy()
        if rna_mask.any() and atac_mask.any():
            return (
                matrix_to_dense(adata.X[:, rna_mask]).astype('float32'),
                matrix_to_dense(adata.X[:, atac_mask]).astype('float32'),
            )
    atac_keys = [key for key in adata.obsm_keys() if any(token in key.lower() for token in ('atac', 'peak'))]
    if atac_keys:
        return matrix_to_dense(adata.X).astype('float32'), matrix_to_dense(adata.obsm[atac_keys[0]]).astype('float32')
    raise RuntimeError('No same-nucleus ATAC matrix or feature block found; stop')

def donor_macro_accuracy(y_true, y_pred, donors):
    frame = pd.DataFrame({'y': y_true, 'pred': y_pred, 'donor': donors})
    per_donor = frame.groupby('donor', observed=True).apply(
        lambda group: float((group.y == group.pred).mean()), include_groups=False
    )
    return float(per_donor.mean())


In [ ]:
def run_real_data():
    import anndata as ad
    path = download_h5ad(DATA_ROOT / f'{DATASET_ID}.h5ad')
    file_summary = inspect_h5ad(path)
    print(json.dumps(file_summary, indent=2))
    if file_summary['pairing_status'] != 'PASS':
        raise RuntimeError('File preflight failed; no model run')
    adata, donor_col, condition_col = sample_cells_by_donor(path)
    x_rna, x_atac = extract_rna_atac(adata)
    labels = (adata.obs[condition_col].astype(str).str.lower() == 'complete trisomy 21').astype(int).to_numpy()
    donors = adata.obs[donor_col].astype(str).to_numpy()
    masks, donor_parts = donor_split(donors, labels, seed=SEED)
    results = []
    for seed in range(SEED, SEED + 5):
        rna_selected, rna_scaler = fit_train_only_view(x_rna, masks['train'], n_features=256)
        atac_selected, atac_scaler = fit_train_only_view(x_atac, masks['train'], n_features=256)
        rna = transform_view(x_rna, rna_selected, rna_scaler)
        atac = transform_view(x_atac, atac_selected, atac_scaler)
        for name, matrix in [('rna_only', rna), ('atac_only', atac), ('concat', np.hstack([rna, atac]))]:
            baseline = LogisticRegression(max_iter=300, class_weight='balanced', random_state=seed)
            baseline.fit(matrix[masks['train']], labels[masks['train']])
            pred = baseline.predict(matrix[masks['test']])
            results.append({'seed': seed, 'model': name, 'intervention': 'normal', **accuracy_report(labels[masks['test']], pred)})
        gated = fit_gated(rna, atac, labels, masks['train'], masks['val'], seed)
        gated.eval()
        xa, xb = torch.tensor(rna[masks['test']]), torch.tensor(atac[masks['test']])
        with torch.no_grad():
            for intervention, kwargs in [
                ('normal', {}),
                ('route_rna_only', {'route_override': [1.0, 0.0]}),
                ('route_atac_only', {'route_override': [0.0, 1.0]}),
                ('ablate_rna', {'ablate': 'rna'}),
                ('ablate_atac', {'ablate': 'atac'}),
            ]:
                logits, weights = gated(xa, xb, **kwargs)
                pred = logits.argmax(1).numpy()
                row = {'seed': seed, 'model': 'gated', 'intervention': intervention, **accuracy_report(labels[masks['test']], pred)}
                row['donor_macro_accuracy'] = donor_macro_accuracy(labels[masks['test']], pred, donors[masks['test']])
                row['mean_rna_route'] = float(weights[:, 0].mean())
                row['mean_atac_route'] = float(weights[:, 1].mean())
                results.append(row)
    frame = pd.DataFrame(results)
    frame.to_csv(OUTPUT_ROOT / 'real_model_results.csv', index=False)
    with (OUTPUT_ROOT / 'real_file_preflight.json').open('w') as handle:
        json.dump(file_summary, handle, indent=2)
    return frame

if RUN_REAL_DATA:
    real_file_summary = inspect_h5ad(download_h5ad(DATA_ROOT / f'{DATASET_ID}.h5ad'))
    print(json.dumps(real_file_summary, indent=2))
    if RUN_MODEL:
        real_results = run_real_data()
        print(real_results.groupby(['model', 'intervention'])[['accuracy', 'donor_macro_accuracy']].mean())
else:
    print('REAL DATA SKIPPED: safe metadata/demo mode')


In [ ]:
final_summary = {
    'metadata_gate': metadata_gate['status'],
    'demo_model_gate': 'PASS',
    'real_file_preflight': 'RUN' if RUN_REAL_DATA else 'PENDING_APPROVAL_AND_LOCAL_DOWNLOAD',
    'real_model': 'RUN' if RUN_MODEL else 'PENDING_APPROVAL_AND_FILE_PREFLIGHT',
    'claim_boundary': 'No disease result in safe mode; demo is synthetic wiring only.',
}
with (OUTPUT_ROOT / 'notebook_summary.json').open('w') as handle:
    json.dump(final_summary, handle, indent=2)
print(json.dumps(final_summary, indent=2))
assert final_summary['metadata_gate'] == 'PASS'
assert final_summary['demo_model_gate'] == 'PASS'
